# Homework Submission - Stage 06: Data Preprocessing

This notebook completes the official starter by generating the provided sample, importing modular cleaning functions, comparing two missing-data strategies, and saving a documented processed dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.cleaning import drop_missing, fill_missing_median, normalize_data

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Generate and load the starter dataset

In [2]:
sample = pd.DataFrame({
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
})
raw_path = RAW / 'sample_data.csv'
sample.to_csv(raw_path, index=False)
df_raw = pd.read_csv(raw_path, dtype={'zipcode': 'string'})
display(df_raw)

,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## 2. Compare missing-data strategies

In [3]:
numeric_columns = ['age', 'income', 'score']

# Main strategy: discard only a mostly-empty field, then impute numeric gaps.
without_sparse_column = drop_missing(df_raw, column_threshold=0.50)
df_filled = fill_missing_median(without_sparse_column, numeric_columns)

# Sensitivity alternative: complete cases for substantive numeric fields.
df_complete_case = drop_missing(df_raw, subset=numeric_columns, column_threshold=0.50)

comparison = pd.DataFrame({
    'original': [len(df_raw), df_raw.isna().sum().sum(), df_raw.shape[1]],
    'median_imputed': [len(df_filled), df_filled.isna().sum().sum(), df_filled.shape[1]],
    'complete_case': [len(df_complete_case), df_complete_case.isna().sum().sum(), df_complete_case.shape[1]],
}, index=['rows', 'missing_cells', 'columns'])
display(comparison)

,original,median_imputed,complete_case
rows,7,7,3
missing_cells,10,0,0
columns,6,5,5


## 3. Normalize numeric fields and validate

In [4]:
df_clean = normalize_data(df_filled, numeric_columns)
scaled_columns = [f'{column}_scaled' for column in numeric_columns]

validation = {
    'no_missing_values': int(df_clean.isna().sum().sum()) == 0,
    'sparse_column_removed': 'extra_data' not in df_clean.columns,
    'raw_numeric_retained': set(numeric_columns).issubset(df_clean.columns),
    'scaled_columns_present': set(scaled_columns).issubset(df_clean.columns),
    'scaled_min_at_least_zero': bool((df_clean[scaled_columns].min() >= 0).all()),
    'scaled_max_at_most_one': bool((df_clean[scaled_columns].max() <= 1).all()),
}
display(pd.Series(validation, name='passed').to_frame())
assert all(validation.values())
display(df_clean)

,passed
no_missing_values,True
sparse_column_removed,True
raw_numeric_retained,True
scaled_columns_present,True
scaled_min_at_least_zero,True
scaled_max_at_most_one,True


,age,income,score,zipcode,city,age_scaled,income_scaled,score_scaled
0,34.0,55000.0,0.820,90210,Beverly,0.238095,0.8125,0.653846
1,45.0,52000.0,0.910,10001,New York,0.761905,0.6250,1.000000
2,29.0,42000.0,0.805,60614,Chicago,0.000000,0.0000,0.596154
3,50.0,58000.0,0.760,94103,SF,1.000000,1.0000,0.423077
4,38.0,52000.0,0.880,73301,Austin,0.428571,0.6250,0.884615
5,39.5,52000.0,0.650,12345,Unknown,0.500000,0.6250,0.000000
6,41.0,49000.0,0.790,94105,San Francisco,0.571429,0.4375,0.538462


## 4. Save the processed dataset and compare summaries

In [5]:
processed_path = PROCESSED / 'sample_data_cleaned.csv'
df_clean.to_csv(processed_path, index=False)

original_summary = df_raw[numeric_columns].describe().T[['count', 'mean', 'std']].add_prefix('original_')
cleaned_summary = df_clean[numeric_columns].describe().T[['count', 'mean', 'std']].add_prefix('cleaned_')
summary_comparison = original_summary.join(cleaned_summary)
display(summary_comparison)
print('Saved:', processed_path)

,original_count,original_mean,original_std,cleaned_count,cleaned_mean,cleaned_std
age,6.0,39.500000,7.556454,7.0,39.500000,6.898067
income,4.0,51000.000000,7071.067812,7.0,51428.571429,5028.490259
score,6.0,0.801667,0.092826,7.0,0.802143,0.084748


Saved: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework06/data/processed/sample_data_cleaned.csv


## 5. Assumptions and reflection

- `extra_data` is removed because 5 of 7 values are missing and the field has no documented business meaning. This avoids inventing mostly synthetic values.
- Median imputation retains all seven rows and is robust to extreme values, but it narrows the apparent distribution and assumes observed values are informative for missing rows.
- Complete-case deletion leaves only three rows, demonstrating why indiscriminate row deletion is costly here. If missingness depends on income or another outcome, either strategy can bias results.
- Scaled columns are added rather than replacing the raw units so the transformation is auditable. In modeling, min/max parameters must be learned on the training fold only to avoid leakage.